In [ ]:
recon_file = ""

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import ptypy

### Set locations for artefacts

In [ ]:
ARTEFACTS = {
  'amplitude': '/tmp/amplitude_plot.png',
  'phase': '/tmp/phase_plot.png',
  'probe_amp': '/tmp/probe_amplitude_plot.png',
  'probe_phase': '/tmp/probe_phase_plot.png',
  'errors': '/tmp/error_plot.png',
}

In [ ]:
if not os.path.exists(recon_file):
    raise ValueError(f"{recon_file} does not exist!")

## Load reconstruction from ptyr file

In [ ]:
with h5py.File(recon_file, "r") as f:
    pixelsize = f['content/obj/Sscan_00G00/_psize'][0]*1e9
    probe = np.array(f['content/probe/Sscan_00G00/data'][:])
    obj = np.array(f['content/obj/Sscan_00G00/data'][0,:,:])
    Niter = len(f['content/runtime/iter_info/'])
    errors = np.array([f["content/runtime/iter_info/%05d/error" %i][:] for i in range(Niter)])
    niters = np.array([f["content/runtime/iter_info/%05d/iteration" %i][...] for i in range(Niter)])
    energy = f["content/probe/Sscan_00G00/_energy"][...]
    wavelength = 1.239/energy
e, v = ptypy.utils.ortho(probe)
print("Pixelsize = %.3f nm" %(pixelsize))
print("Photon energy = %f" %(energy*1000))

In [ ]:
plt.figure(figsize=(8,4), dpi=100)
plt.plot(niters, errors[:,1])
plt.ylabel("Log-likelihood")
plt.xlabel("Nr. of iterations")
plt.savefig(ARTEFACTS['errors'])
plt.show()

## Plot amplitude and phase histograms

In [ ]:
plt.hist(np.abs(obj).flatten(), bins=100)
plt.show()

In [ ]:
plt.hist(np.angle(obj).flatten(), bins=100)
plt.show()

## Plot amplitude

In [ ]:
plt.figure(figsize=(6,5),dpi=100)
plt.axis("off")
plt.imshow(np.abs(obj), cmap="gray", interpolation='none',vmin=0, vmax=5)
plt.gca().add_patch(plt.Rectangle((10,10),1000/pixelsize,5, color='w'))
plt.gca().text(10+1000/pixelsize/2,20,r"%d um" %(1000/1e3), color='w', va='top', ha='center', fontsize=15)
plt.colorbar()
plt.savefig(ARTEFACTS['amplitude'])
plt.show()

## Plot phase

In [ ]:
plt.figure(figsize=(6,5), dpi=100)
plt.axis("off")
plt.imshow(np.angle(obj), cmap="gray", vmin=-1*np.pi, vmax=1*np.pi, interpolation='none')
plt.gca().add_patch(plt.Rectangle((10,10),1000/pixelsize,5, color='w'))
plt.gca().text(10+1000/pixelsize/2,20,r"%d μm" %(1000/1e3), color='w', va='top', ha='center', fontsize=15)
plt.colorbar()
plt.savefig(ARTEFACTS['phase'])
plt.show()

## Plot probe

In [ ]:
m = 0
fig = plt.figure(figsize=(5,5))
ax = ptypy.utils.PtyAxis(fig.add_subplot(111), channel='a', vmin=None, vmax=None)
ax.set_data(np.abs(v[m]))
ax.ax.axis("off")
ax.ax.add_patch(plt.Rectangle((5,5),1000/pixelsize,10, color='w'))
ax.ax.text(5+1000/pixelsize/2,20,r"%d μm" %(1000/1e3), color='w', va='top', ha='center', fontsize=20)
ax.ax.set_title(f"Probe Amplitude | {e[m]*100:.1f} %")
fig.savefig(ARTEFACTS['probe_amp'])
plt.show()

In [ ]:
m = 0
fig = plt.figure(figsize=(5,5))
ax = ptypy.utils.PtyAxis(fig.add_subplot(111), channel='a', vmin=None, vmax=None)
ax.set_data(np.angle(v[m]))
ax.ax.set_title(f"Probe Phase | {e[m]*100:.1f}")
ax.ax.axis("off")
fig.savefig(ARTEFACTS['probe_phase'])
plt.show()

## Plot propagation

In [ ]:
import ptypy.utils as u
from ptypy.core import geometry

def nfprop(energy, distance, resolution, shape):
    g = u.Param()
    g.energy = energy
    g.distance = distance
    g.psize = resolution
    g.shape = shape
    g.propagation = "nearfield"
    G = geometry.Geo(owner=None, pars=g)
    return G.propagator.fw

def propagated(probe, distances, energy, resolution, eps=1e-9):
    prop_array = np.zeros((len(distances),)+probe.shape, dtype=probe.dtype)
    for i,d in enumerate(distances):
        if np.abs(d) < eps:
            prop_array[i] = probe.copy()
        else:
            prop_array[i] = nfprop(energy, d, resolution, probe.shape)(probe)
    return prop_array

In [ ]:
prb = v[0]
full_defocus_range = np.linspace(-50e-6, 50e-6, 201)
main_probe = propagated(prb, full_defocus_range, energy, pixelsize*1e-9)

In [ ]:
cz,cy,cx = np.unravel_index(np.abs(main_probe).argmax(),main_probe.shape)
ny,nx = prb.shape
fz,fy,fx = 100,cy,cx

In [ ]:
fig = plt.figure(figsize=(12,4),dpi=100)
axy = fig.add_subplot(211)
axx = fig.add_subplot(212)
axz = fig.add_axes([-0.2,0.025, 0.3,0.85])

Axz = ptypy.utils.PtyAxis(axz, channel='a', vmax=0.25*np.abs(main_probe).max())
Axz.set_data(main_probe[fz])
axz.axvline(fx, color='w', ls=':')
axz.axhline(fy, color='w', ls=':')
axz.axis('off')
axz.add_patch(plt.Rectangle((5,5),1000/pixelsize,10, color='w'))
axz.text(5+1000/pixelsize/2,20,r"%d μm" %(1000/1e3), color='w', va='top', ha='center', fontsize=10)

Axy = ptypy.utils.PtyAxis(axy, channel='a', vmax=0.5*np.abs(main_probe).max())
Axy.set_data(main_probe[:,fy].T)
axy.set_aspect('auto')
axy.axvline(fz, color='w', ls=':')
axy.axvline(cz, color='r', ls=':')
axy.set_xticks([])
axy.set_yticks(range(ny)[5::50])
axy.set_yticklabels(np.round(np.linspace(-cy*pixelsize/1e3, cy*pixelsize/1e3,nx)[5::50],1))
axy.tick_params('y', right=True, left=False, labelright=True, labelleft=False)
axy.set_ylabel("horiz. position [μm]")
axy.yaxis.set_label_position("right")

Axx = ptypy.utils.PtyAxis(axx, channel='a', vmax=0.5*np.abs(main_probe).max())
Axx.set_data(main_probe[:,:,fx].T)
axx.axvline(fz, color='w', ls=':')
axx.axvline(cz, color='r', ls=':')
axx.set_aspect('auto')
axx.set_xticks(range(len(full_defocus_range))[::20])
axx.set_xticklabels(np.round(full_defocus_range[::20] * 1e6).astype(int))
axx.set_yticks(range(ny)[5::50])
axx.set_yticklabels(np.round(np.linspace(-cx*pixelsize/1e3, cx*pixelsize/1e3,nx)[5::50],1))
axx.tick_params('y', right=True, left=False, labelright=True, labelleft=False)
axx.set_xlabel("Defocus [μm]")
axx.set_ylabel("vert. position [μm]")
axx.yaxis.set_label_position("right")

plt.show()